# Tier 2 — z-score on store×month 5%

After Tier 1 MVP (blacklist + freq store-day), flag store-months with top-box rate outside network mean ± 3σ.

In [ ]:
##Шаг A — загрузка

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()
from db.connection import get_engine
from fraud_guard.tier1 import (
    Tier1Config, apply_tier1, filter_answered_metric_rows,
    keep_clean_rows, top_box_rate,
)

engine = get_engine()
df = pd.read_sql("SELECT * FROM dbo.TargetsByMetrics_RateGetAnswers", engine)
print("shape:", df.shape)

shape: (1213474, 35)


In [ ]:
##Шаг B — Tier 1 MVP (база для Tier 2)

In [2]:
work = filter_answered_metric_rows(df)
flagged = apply_tier1(work, config=Tier1Config(enable_always_topbox=False))
clean = keep_clean_rows(flagged)

print("answered:", len(work), "top-box:", round(top_box_rate(work), 4))
print("after Tier1 MVP:", len(clean), "top-box:", round(top_box_rate(clean), 4))

answered: 959170 top-box: 0.681
after Tier1 MVP: 925083 top-box: 0.6712


In [ ]:
##Шаг C — панель store×month + z-score

#z>3 сверху = 0 (порог 101% нереален)
#z>2 = 138, ≥90% = 135 — вот рабочие кандидаты
#z<−3 = 5 — слишком низкий score, не fraud

In [3]:
def five_pct(s):
    return 100.0 * (s == 5).mean()

panel = (
    clean.groupby(["PrintStore", "Year", "Month"], dropna=False)["Answer_Value"]
    .agg(volume="count", five_pct=five_pct)
    .reset_index()
)
panel = panel[panel["volume"] >= 30].copy()

mu = panel["five_pct"].mean()
sigma = panel["five_pct"].std(ddof=0)
panel["z"] = (panel["five_pct"] - mu) / sigma

print("panel rows (vol>=30):", len(panel))
print("mean:", round(mu, 2), "std:", round(sigma, 2))
print("mean+3*std:", round(mu + 3 * sigma, 2), "← если >100, z>3 сверху почти невозможен")
print("z > 3:", int((panel["z"] > 3).sum()))
print("z > 2:", int((panel["z"] > 2).sum()))
print("five_pct >= 90:", int((panel["five_pct"] >= 90).sum()))
print("z < -3:", int((panel["z"] < -3).sum()))
print(panel["z"].describe(percentiles=[0.5, 0.95, 0.99]))

panel rows (vol>=30): 4555
mean: 66.55 std: 11.54
mean+3*std: 101.17 ← если >100, z>3 сверху почти невозможен
z > 3: 0
z > 2: 138
five_pct >= 90: 135
z < -3: 5
count    4.555000e+03
mean     8.111575e-17
std      1.000110e+00
min     -3.160265e+00
50%     -3.589462e-02
95%      1.753248e+00
99%      2.315907e+00
max      2.898374e+00
Name: z, dtype: float64


In [ ]:
## Шаг D — кто эти «высокие» store-months

In [4]:
high = panel[(panel["z"] > 2) | (panel["five_pct"] >= 90)].sort_values("five_pct", ascending=False)
print("high candidates:", len(high))
print(high.head(15).to_string(index=False))
print("\nvolume in high:", int(high["volume"].sum()),
      "share of panel volume:", round(high["volume"].sum() / panel["volume"].sum(), 4))

high candidates: 138
 PrintStore  Year  Month  volume   five_pct        z
      154.0  2026      3      31 100.000000 2.898374
      154.0  2026      5      66  98.484848 2.767082
      240.0  2026      8     220  98.181818 2.740824
      240.0  2026      7     211  98.104265 2.734104
      280.0  2026      8      37  97.297297 2.664178
      280.0  2025      8     186  96.774194 2.618850
      144.0  2026      7     179  96.648045 2.607919
      154.0  2025      8      59  96.610169 2.604637
      275.0  2026      1      87  96.551724 2.599572
      306.0  2026      4     111  96.396396 2.586113
      262.0  2026      7     108  96.296296 2.577439
      154.0  2026      1      52  96.153846 2.565095
      306.0  2025      7     209  95.693780 2.525229
       43.0  2026      7     852  95.657277 2.522066
      144.0  2025      5     304  95.394737 2.499316

volume in high: 24920 share of panel volume: 0.027


In [ ]:
## Шаг E — эффект на сеть (используем ваш high)

In [5]:
high_keys = set(zip(high["PrintStore"], high["Year"], high["Month"]))
mask = [
    (s, y, m) in high_keys
    for s, y, m in zip(clean["PrintStore"], clean["Year"], clean["Month"])
]

after_t2 = clean.loc[[not x for x in mask]].copy()

print("before (Tier1):", len(clean), "top-box:", round(top_box_rate(clean), 4))
print("dropped rows:", int(sum(mask)), "share:", round(sum(mask) / len(clean), 4))
print("after Tier2:", len(after_t2), "top-box:", round(top_box_rate(after_t2), 4))
print("top-box in dropped:", round(top_box_rate(clean.loc[mask]), 4))
print("network delta pp:", round(100 * (top_box_rate(after_t2) - top_box_rate(clean)), 2))

before (Tier1): 925083 top-box: 0.6712
dropped rows: 24920 share: 0.0269
after Tier2: 900163 top-box: 0.6641
top-box in dropped: 0.927
network delta pp: -0.71


In [ ]:
## Шаг F — итоговая сводка pipeline

In [6]:
print("=== pipeline ===")
print("0 answered Q10012: ", len(work), " top-box:", round(top_box_rate(work), 4))
print("1 Tier1 MVP:       ", len(clean), " top-box:", round(top_box_rate(clean), 4))
print("2 Tier2 high z/90: ", len(after_t2), " top-box:", round(top_box_rate(after_t2), 4))
print("total dropped:", len(work) - len(after_t2),
      "share:", round((len(work) - len(after_t2)) / len(work), 4))
print("total delta pp:", round(100 * (top_box_rate(after_t2) - top_box_rate(work)), 2))

=== pipeline ===
0 answered Q10012:  959170  top-box: 0.681
1 Tier1 MVP:        925083  top-box: 0.6712
2 Tier2 high z/90:  900163  top-box: 0.6641
total dropped: 59007 share: 0.0615
total delta pp: -1.68
